# 🩺 Nidaan — YOLOv8 Prescription Line Detector Training

**Purpose:** Trains YOLOv8-Nano to detect medicine line bounding boxes on full prescription images.

**Before running:**
1. `Runtime` → `Change runtime type` → **T4 GPU**
2. Upload `prescription_images.zip` (your `datasets/images/` folder zipped) via Files panel
3. **Important:** Complete Label Studio annotation first (see `scripts/annotate_guide.md`)
4. Upload `annotations.zip` (exported from Label Studio in YOLO format)
5. `Runtime` → `Run all`

**Expected time:** ~45 minutes on T4

In [ ]:
# ── CELL 1: Install dependencies ────────────────────────────────────────
!pip install -q ultralytics
from ultralytics import YOLO
import torch
print(f'✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'✅ Ultralytics YOLO ready')

In [ ]:
# ── CELL 2: Unzip prescription images and annotations ───────────────────
import os, zipfile

for zname, dest in [
    ('prescription_images.zip', '/content/rx_images'),
    ('annotations.zip', '/content/annotations'),
]:
    if os.path.exists(f'/content/{zname}'):
        print(f'Extracting {zname}...')
        with zipfile.ZipFile(f'/content/{zname}') as z:
            z.extractall(dest)
        print(f'✅ {zname} extracted')
    else:
        print(f'⚠ {zname} not found — upload it first')

n_images = len([f for f in os.listdir('/content/rx_images') if f.endswith(('.jpg','.jpeg','.png'))])
print(f'\nPrescription images found: {n_images}')

In [ ]:
# ── CELL 3: Create YOLO dataset structure ───────────────────────────────
import os, shutil, random

BASE = '/content/yolo_rx'
for split in ['train', 'val']:
    os.makedirs(f'{BASE}/{split}/images', exist_ok=True)
    os.makedirs(f'{BASE}/{split}/labels', exist_ok=True)

images = sorted([f for f in os.listdir('/content/rx_images')
                 if f.lower().endswith(('.jpg','.jpeg','.png'))])
random.seed(42)
random.shuffle(images)
split_idx = int(len(images) * 0.85)  # 85% train, 15% val
train_imgs = images[:split_idx]
val_imgs   = images[split_idx:]

for fname, split in [(f,'train') for f in train_imgs] + [(f,'val') for f in val_imgs]:
    # Copy image
    shutil.copy(f'/content/rx_images/{fname}', f'{BASE}/{split}/images/{fname}')
    # Copy label (same name, .txt extension)
    label = os.path.splitext(fname)[0] + '.txt'
    lpath = f'/content/annotations/{label}'
    if os.path.exists(lpath):
        shutil.copy(lpath, f'{BASE}/{split}/labels/{label}')

print(f'✅ Train: {len(train_imgs)} | Val: {len(val_imgs)}')

# Write YAML config
yaml_content = f"""path: {BASE}
train: train/images
val: val/images
nc: 1
names: ['medicine_line']
"""
with open('/content/rx_dataset.yaml', 'w') as f:
    f.write(yaml_content)
print('✅ YAML config written')

In [ ]:
# ── CELL 4: Train YOLOv8-Nano layout detector ───────────────────────────
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # Nano — smallest, fastest for CPU inference

print('⚡ Training YOLOv8 prescription line detector...')
results = model.train(
    data='/content/rx_dataset.yaml',
    epochs=100,
    imgsz=1024,
    batch=8,
    device='0',           # T4 GPU
    patience=20,          # Early stopping
    save=True,
    project='/content/yolo_rx_training',
    name='run1',
    exist_ok=True,
    workers=2,
    verbose=True,
)

In [ ]:
# ── CELL 5: Evaluate and download model ─────────────────────────────────
import shutil, os

best_pt = '/content/yolo_rx_training/run1/weights/best.pt'
print('📊 Validation results:')
val_results = model.val(data='/content/rx_dataset.yaml', weights=best_pt)
print(f'   mAP@0.5: {val_results.box.map50:.3f}')
print(f'   mAP@0.5:0.95: {val_results.box.map:.3f}')

shutil.copy(best_pt, '/content/yolov8_rx_lines.pt')
print(f'\n✅ Best model saved: yolov8_rx_lines.pt')

from google.colab import files
files.download('/content/yolov8_rx_lines.pt')
print('⬇️ Downloading... Extract to D:\\Nidaan\\models\\ on your laptop.')